# Autoresearch Test Notebook
Verifies prompt loading, stage configs, and SR metric work correctly.
Does NOT require GPU — no model loading.

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('')), 'src'))
print('Setup OK')

Setup OK


## Test 1: Tower of Hanoi — YAML prompt loading

In [2]:
from tower_of_hanoi.enviroment import TowerOfHanoi
from tower_of_hanoi import prompts as toh_prompts

game = TowerOfHanoi(num_disks=3)

for variant in ['base', 'cot_detailed', 'minimal']:
    sys_prompt = toh_prompts.get_system_prompt(game, variant=variant)
    usr_prompt = toh_prompts.build_user_prompt(
        current_state=game.get_state(),
        previous_move='None',
        environment=game,
        step=1,
        variant=variant,
    )
    assert '{num_disks}' not in sys_prompt, f'Unfilled placeholder in {variant} system prompt'
    assert '{current_state}' not in usr_prompt, f'Unfilled placeholder in {variant} user prompt'
    print(f'  {variant}: OK (sys={len(sys_prompt)} chars, usr={len(usr_prompt)} chars)')

print('PASS: All ToH prompt variants load and format correctly')

  base: OK (sys=1151 chars, usr=822 chars)
  cot_detailed: OK (sys=745 chars, usr=516 chars)
  minimal: OK (sys=214 chars, usr=265 chars)
PASS: All ToH prompt variants load and format correctly


## Test 2: Sliding Puzzle — YAML prompt loading

In [3]:
from sliding_puzzle.enviroment import SlidingPuzzle, CONFIGS
from sliding_puzzle import prompts as sp_prompts

game = SlidingPuzzle(initial_state=CONFIGS['3x3 (easiest)'])

for variant in ['base', 'cot_detailed', 'minimal']:
    sys_prompt = sp_prompts.get_system_prompt(game, variant=variant)
    usr_prompt = sp_prompts.build_user_prompt(
        current_state=game.get_state(),
        previous_move='None',
        environment=game,
        step=1,
        variant=variant,
    )
    assert '{n}' not in sys_prompt, f'Unfilled placeholder in {variant} system prompt'
    assert '{current_state}' not in usr_prompt, f'Unfilled placeholder in {variant} user prompt'
    print(f'  {variant}: OK (sys={len(sys_prompt)} chars, usr={len(usr_prompt)} chars)')

print('PASS: All SP prompt variants load and format correctly')

  base: OK (sys=2426 chars, usr=366 chars)
  cot_detailed: OK (sys=1249 chars, usr=428 chars)
  minimal: OK (sys=197 chars, usr=123 chars)
PASS: All SP prompt variants load and format correctly


## Test 3: All CONFIGS create valid SlidingPuzzle instances

In [4]:
from sliding_puzzle.enviroment import SlidingPuzzle, CONFIGS

for name, state in CONFIGS.items():
    puzzle = SlidingPuzzle(initial_state=state)
    sr = puzzle.compute_progress()
    print(f'  {name}: size={puzzle.grid_size}x{puzzle.grid_size}, SR={sr:.1%}, goal={puzzle.goal_state[:4]}...')

print('PASS: All CONFIGS create valid puzzles')

  2x2: size=2x2, SR=25.0%, goal=[0, 1, 2, 3]...
  3x3 (easiest): size=3x3, SR=0.0%, goal=[0, 1, 2, 3]...
  3x3 (hardest): size=3x3, SR=0.0%, goal=[0, 1, 2, 3]...
  4x4 (easiest): size=4x4, SR=0.0%, goal=[0, 1, 2, 3]...
  4x4 (hardest): size=4x4, SR=0.0%, goal=[0, 1, 2, 3]...
PASS: All CONFIGS create valid puzzles


## Test 4: SR (compute_progress) for both games

In [5]:
from sliding_puzzle.enviroment import SlidingPuzzle, CONFIGS
from tower_of_hanoi.enviroment import TowerOfHanoi

# Sliding Puzzle: goal is [0,1,2,...,8]
# State [0,1,2,3,4,5,6,7,8] = goal = solved
p = SlidingPuzzle(initial_state=CONFIGS["2x2"])  # [2,1,3,0], goal=[0,1,2,3]
print(f"SP 2x2 initial: state={p.get_state()}, goal={p.goal_state}, SR={p.compute_progress():.1%}")

# Check a partially correct state
p2 = SlidingPuzzle(initial_state=CONFIGS["3x3 (easiest)"])
sr = p2.compute_progress()
print(f"SP 3x3 easiest: state={p2.get_state()}, SR={sr:.1%}")

# Tower of Hanoi: 0%
h = TowerOfHanoi(num_disks=3)
assert h.compute_progress() == 0.0
print(f"ToH initial: SR=0%")

# Tower of Hanoi: solved
h2 = TowerOfHanoi(num_disks=3)
for m in [(1,0,2), (2,0,1), (1,2,1), (3,0,2), (1,1,0), (2,1,2), (1,0,2)]:
    h2.move_disk(*m)
assert h2.compute_progress() == 1.0
print("ToH solved: SR=100%")

print("PASS: compute_progress works for both games")

SP 2x2 initial: state=[2, 1, 3, 0], goal=[0, 1, 2, 3], SR=25.0%
SP 3x3 easiest: state=[1, 2, 5, 6, 3, 4, 7, 8, 0], SR=0.0%
ToH initial: SR=0%
ToH solved: SR=100%
PASS: compute_progress works for both games


## Test 5: Autoresearch dry-run

In [6]:
import subprocess
result = subprocess.run(
    ['uv', 'run', 'scripts/autoresearch.py', '--game', 'tower_of_hanoi', '--dry-run'],
    capture_output=True, text=True, cwd=os.path.join(os.path.dirname(os.path.abspath(''))),
)
print(result.stdout)
assert result.returncode == 0, f'Dry run failed: {result.stderr}'
assert 'Total param combos' in result.stdout
print('PASS: autoresearch dry-run works')

Game: tower_of_hanoi
Stages: [3, 4, 5, 6, 7, 8, 9, 10]
Total param combos: 486
Time budget: 23.5h

First 5 experiments:
  1. {'model_name': 'qwen3-32b', 'model_path': 'LLM/models/qwen3-32b', 'temperature': 0.0, 'prompt_variant': 'base', 'max_agents_per_step': 1, 'margin_k': 1}
  2. {'model_name': 'qwen3-32b', 'model_path': 'LLM/models/qwen3-32b', 'temperature': 0.0, 'prompt_variant': 'base', 'max_agents_per_step': 1, 'margin_k': 2}
  3. {'model_name': 'qwen3-32b', 'model_path': 'LLM/models/qwen3-32b', 'temperature': 0.0, 'prompt_variant': 'base', 'max_agents_per_step': 1, 'margin_k': 3}
  4. {'model_name': 'qwen3-32b', 'model_path': 'LLM/models/qwen3-32b', 'temperature': 0.0, 'prompt_variant': 'base', 'max_agents_per_step': 3, 'margin_k': 1}
  5. {'model_name': 'qwen3-32b', 'model_path': 'LLM/models/qwen3-32b', 'temperature': 0.0, 'prompt_variant': 'base', 'max_agents_per_step': 3, 'margin_k': 2}

PASS: autoresearch dry-run works


## Summary
All tests passed:
- **Test 1**: ToH YAML prompt variants load and format correctly
- **Test 2**: SP YAML prompt variants load and format correctly
- **Test 3**: All CONFIGS create valid SlidingPuzzle instances
- **Test 4**: compute_progress() works for both games
- **Test 5**: autoresearch.py dry-run executes without errors